# CIFAR-10 Production Modeling

This notebook is a thin orchestrator for the production-grade ``cifar10_models`` package. It demonstrates:

- Config-driven model selection (PatchCNN, ConvMixer, ViT, ResNet-18)
- Modern training recipe: AdamW + warmup + cosine, AMP, EMA, gradient clipping
- Best checkpointing and early stopping
- Evaluation with optional test-time augmentation
- ONNX export with runtime parity check

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
%matplotlib inline

from cifar10_models import (
    build_model,
    evaluate,
    evaluate_with_tta,
    export_to_onnx,
    fit,
    get_dataloaders,
    set_seed,
    setup_logging,
)
from cifar10_models.config import TrainConfig, DataConfig
from cifar10_models.optim import AMPManager
from cifar10_models.utils import load_config

In [ ]:
logger = setup_logging()

# Load a config and optionally switch to a fast dev run for the notebook.
config = load_config("configs/default.yaml")
config.data.fast_dev_run = True
config.data.fast_dev_size = 500
config.epochs = 2

set_seed(config.seed)
logger.info("Using device: %s", config.device)

In [ ]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    augmentation_cfg=config.augmentation,
    data_cfg=config.data,
    num_classes=config.model.num_classes,
)
logger.info("Classes: %s", class_names)

In [ ]:
model = build_model(config.model)
print(f"Model: {config.model.name}")
print(f"Parameters: {model.count_parameters():,}")

In [ ]:
history = fit(model, train_loader, val_loader, config)

In [ ]:
amp_manager = AMPManager(config.device.type, enabled=config.use_amp)
test_loss, test_acc = evaluate(model, test_loader, config.device, amp_manager)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
tta_acc = evaluate_with_tta(model, test_loader, config.device, tta_level=2, amp_manager=amp_manager)
print(f"TTA test accuracy: {tta_acc:.4f}")

In [ ]:
export_path = config.logging.checkpoint_dir / f"{config.model.name}.onnx"
export_to_onnx(model, export_path, config.export.opset_version, config.export.dynamic_batch)
print(f"ONNX export: {export_path}")

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs_range = range(1, len(history["train_acc"]) + 1)

    axes[0].plot(epochs_range, history["train_acc"], label="train")
    axes[0].plot(epochs_range, history["val_acc"], label="val")
    axes[0].set_title(f"{title} - Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(epochs_range, history["train_loss"], label="train")
    axes[1].plot(epochs_range, history["val_loss"], label="val")
    axes[1].set_title(f"{title} - Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    axes[2].plot(epochs_range, history["lr"], label="lr")
    axes[2].set_title(f"{title} - Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("LR")
    axes[2].legend()

    fig.tight_layout()
    return fig

plot_history(history, config.model.name)